<img src="media/Oc10-Banner_waterlevel+esalogo.png" width="100%" alt="Banner for EO AFRICA course" />


*R. Rietbroek, May 2026*

<font color=#cf0072>

# Terrestrial Water Storage tutorial
</font>

In [ ]:
%%bash
# you only need to execute the following once for your profile (when working on the Innovation lab)
# after that you can use the "pytws" notebook kernel (See top right of the notebook)
source ~/.bashrc
conda activate /home/eoafrica/shared/w3-4/pytws
python -m ipykernel install --user --name="pytws"



<font color=#cf0072>
    
## Introduction and goal of the tutorial
The goal of this tutorial is to extract a Terrestrial Water Storage map from GRACE and/or GRACE-FO and a time series of a location of choice. Note that filtering is required.

Terrestrial Water Storage, in meters of equivalent water height, at a certain location and time can be computed from the Level 2 Stokes (gravity) coefficients using the following formula:
$$TWS (\theta, \lambda ) = a\sum_{n=0}^{\infty}\sum_{m=0}^{m=n}\frac{2n+1}{1+k_n }\frac{\rho_e}{\rho_w} \Delta C_{nm} Y_{nm}(\theta,\lambda) $$

Note that in order to get reasonable results, the Stokes coefficient anomalies are generated by subtracting a static gravity field model, followed by an (optional) filtering method, to reduce the increasing noise for coefficients with high degrees (n). 

The necesary code for this is implemented in [shxarray](https://github.com/ITC-Water-Resources/shxarray). On the Innovation lab, a conda environment has been prepared already. Refer to [This notebook to get you started on a different Jupyter instance](./00_OC10-Downloaddata.ipynb). Furthermore, the GRACE/GARCE-FO data from the TU GRAZ university needed has also been downloaded already.


When you've made a copy of this notebook, it's yours and  you can modify it to your liking. We recommend adding Readme cells where you elaborate on the lessons learned in your own words. We've added some steps to invite you to make changes to your notebook.

</font>


<font color=#cf0072>

## Step 1 (~500 words): Introduction
1. Write a short introduction on the purpose of this notebook aimed at your fellow pees.
2. Provide a flow chart of its working

### Hints & Tips
* You can create a flowchart with an external program and embed it in the markdown cell below

</font>

.. YOUR ANSWER ...

<font color=#cf0072>

## Step 2: Load the necessary modules

### Hints & Tips
* Are all modules, needed for this exercise, present? (i.e. you can restart and rerun the entire exercise in the specified order to check)
* Besides the standard modules (os etc.) do you motivate why some special modules are needed?

</font>


In [ ]:
## ANSWER MODEL STEP2
import xarray as xr
# we need shxarray to perform computations on the Stokes coefficients from GRACE
import shxarray
from glob import glob
import numpy as np
import os

import matplotlib.pyplot as plt

<font color=#cf0072>

## Step 3: generate a list of the available GRACE files and sorte them chronologically

</font>


In [ ]:
dataroot="data"
gfopath='ITSG-Grace_operational/monthly/monthly_n96/'
grcpath='ITSG-Grace2018/monthly/monthly_n96/'

grcfiles=[f for f in glob(os.path.join(dataroot,grcpath)+"/ITSG*gfc")]

grcfiles.extend([f for f in glob(os.path.join(dataroot,gfopath)+"/ITSG*gfc")])

# note in this case the lexical sort is also conveniently the chronological sort
grcfiles=sorted(grcfiles)

staticfile=os.path.join(dataroot,'GOCO06s.gfc')




<font color=#cf0072>

## Step 4: Load the GRACE and GRACE-FO solutions, substract a static gravity field (GOCO06S) catchment polygon, and convert it to terrestrial water storage


</font>


In [ ]:
## ANSWER MODEL STEP 4
nmax=96
nmin=2 # note for this exercise we ignore degree 1 coefficients, for production applications, external degree 1 coefficients are necessary

dsgrace=xr.open_mfdataset(grcfiles,engine='icgem').sh.build_nmindex().sh.truncate(nmax,nmin)

# load static gravity field
dsgravstatic=xr.open_dataset(staticfile,engine='icgem').sh.build_nmindex().sh.truncate(nmax,nmin)

In [ ]:
## ANSWER MODEL STEP 4
dadeltacnm=(dsgrace.cnm-dsgravstatic.cnm)
datws=dadeltacnm.sh.tws(ingravtype='stokes').compute()

<font color=#cf0072>

## Step 5: Convert a specific time slice the TWS coefficients it a grid and plot
</font>


In [ ]:
timeselect=np.datetime64('2005-10-01')

datwsgrid=datws.sel(time=timeselect,method='nearest').sh.synthesis()



In [ ]:
#quick plot
vmax=0.5 # 50 cm
vmin=-vmax
datwsgrid.plot(vmin=vmin,vmax=vmax)

<font color=#cf0072>

## Step 6: Woah! That doesn't look quite allright, can we get rid of those north-south stripes?
Yes, you can but it comes at a trade-off. A filter (e.g. Gaussian filter, or a specialized grace filter) can be applied, but the spatial resolution of the signal will decrease.
So let's try out some filters

</font>


In [ ]:
## ANSWER MODEL STEP 6
#filter witha Gaussian kernel with a halfwidth radius of 500km
datwsgridgauss=datws.sel(time=timeselect,method='nearest').sh.filter('Gauss500').sh.synthesis()

datwsgridddk=datws.sel(time=timeselect,method='nearest').sh.filter('DDK6').sh.synthesis()


In [ ]:
import cartopy.crs as ccrs
import matplotlib.pyplot as plt

ax = plt.axes(projection=ccrs.PlateCarree())
ax.coastlines()
datwsgridgauss.plot(ax=ax,vmin=vmin,vmax=vmax,cbar_kwargs=dict(orientation='horizontal',label='TWS[m]'))
plt.show()



In [ ]:
ax = plt.axes(projection=ccrs.PlateCarree())
ax.coastlines()
datwsgridddk.plot(ax=ax,vmin=vmin,vmax=vmax,cbar_kwargs=dict(orientation='horizontal',label='TWS[m]'))
plt.show()

<font color=#cf0072>

## Step 7 (~500 words): Reflect on the found  GRACE/GRACE-FO TWS and the strength of the filters
</font>


... YOUR ANSWER ....